# Prompt robustness on raw text

Run the setup cell first, then review model/checkpoint paths before GPU evaluation.
The main held-out rows are matched by exact `input_text` and `emotion` to the
main CSV's original `text`. Each alternative prompt is applied once to that raw
sentence. Split membership, row order, duplicate rows, and labels are preserved.
Missing or ambiguous matches raise an error instead of silently changing data.

This corrects nested prompts in the supplied notebook. Corrected outputs and
paper-table exports use `data/prompt_robustness_evaluation_raw_text/`; earlier
scores must be rerun before they can be attributed to this corrected workflow.
CPU input-construction checks passed; GPU scores have not been reproduced.
See `results/validation/prompt_robustness_inputs.json` and `docs/RELEASE_REVIEW.md`.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
from __future__ import annotations
"""
Evaluate prompt robustness for vanilla and best SAE-steered models.

Models:
    - google/gemma-2-2b
    - google/gemma-2-9b
    - meta-llama/Llama-3.1-8B

The script evaluates the original prompt plus three prompt variations on the
same fixed test dataset. For each model and prompt, it compares:
    1. vanilla model predictions, with no SAE hooks;
    2. predictions using the best saved feature-steering checkpoint.

Expected files for each model:
    <feature_dir>/auto_selected_unique_features_with_affected_emotions.csv
    <steering_run_dir>/best_feature_steering.pt

Run from the directory that contains feature_steering_utils.py:
    python evaluate_prompt_robustness.py
"""

import os
from typing import Callable, List, Optional


def setup_hf_cache(cache_root: str = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))):
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_DATASETS_CACHE"] = f"{cache_root}/datasets"
    os.environ["TRANSFORMERS_CACHE"] = f"{cache_root}/transformers"
    os.environ["HUGGINGFACE_HUB_CACHE"] = f"{cache_root}/hub"
    os.environ["TORCH_HOME"] = f"{cache_root}/torch"
    os.environ["XDG_CACHE_HOME"] = cache_root


setup_hf_cache()



import gc
import traceback
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sae_emotion.feature_steering_utils import (
    DEFAULT_EMOTIONS,
    SAEFeatureFinetuner,
    prepare_emotion_dataframe,
    safe_save_csv,
)

In [ ]:
# ============================================================
# User settings
# ============================================================

# External datasets evaluated with every prompt.
EXTERNAL_TEST_DATASETS = [
    {
        "name": "xed_en",
        "path": "data/xed-en.csv",
        "text_col": "text",
        "label_col": "emotion",
        "sep": ",",
        "drop_labels": None,
    },
    {
        "name": "emoevent_en",
        "path": "data/emoevent_en.csv",
        "text_col": "tweet",
        "label_col": "emotion",
        "sep": "\t",
        "drop_labels": ["others"],
    },
]

# Use the author-supplied shared holdout and verify every checkpoint run
# saved exactly the same rows and row order before evaluation.
MAIN_TEST_DATASET_NAME = "main_test"
MAIN_TEST_FILENAME = "val_df.csv"
MAIN_TEST_PATH = "data/val_df.csv"
MAIN_SOURCE_PATH = "data/self_reported_ekman_zeroshot_preds.csv"

OUTPUT_DIR = "data/prompt_robustness_evaluation_raw_text"
FEATURE_CSV_NAME = "auto_selected_unique_features_with_affected_emotions.csv"

# This should match the run directory produced by the training script.
# The directory name in the existing training script says lr_5e-3 even though
# the configured learning rate is 1e-2.
STEERING_RUN_DIR_NAME = "lr_5e-3_l1_0p0"

CACHE_DIR: Optional[str] = None
DEVICE = "cuda"
DEVICE_MAP = "cuda"
TORCH_DTYPE = torch.bfloat16
EVAL_BATCH_SIZE = 16
MAX_LENGTH = 512

SAVE_PREDICTIONS = True


@dataclass(frozen=True)
class ModelSpec:
    model_name: str
    feature_dir: str
    sae_size: Optional[str]

    @property
    def feature_csv_path(self) -> str:
        return os.path.join(self.feature_dir, FEATURE_CSV_NAME)

    @property
    def steering_run_dir(self) -> str:
        return os.path.join(
            self.feature_dir,
            "global_feature_steering",
            STEERING_RUN_DIR_NAME,
        )

    @property
    def best_checkpoint_path(self) -> str:
        return os.path.join(self.steering_run_dir, "best_feature_steering.pt")

    @property
    def main_test_path(self) -> str:
        return os.path.join(self.steering_run_dir, MAIN_TEST_FILENAME)


MODEL_SPECS = [
    ModelSpec(
        model_name="google/gemma-2-2b",
        feature_dir="data/google-gemma-2-2b-feature-act",
        sae_size=None,
    ),
    ModelSpec(
        model_name="google/gemma-2-9b",
        feature_dir="data/google-gemma-2-9b-feature-act",
        sae_size=None,
    ),
    ModelSpec(
        model_name="google/gemma-3-4b-pt",
        feature_dir="data/google-gemma-3-4b-pt-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="google/gemma-3-4b-it",
        feature_dir="data/google-gemma-3-4b-it-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="google/gemma-3-12b-pt",
        feature_dir="data/google-gemma-3-12b-pt-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="google/gemma-3-12b-it",
        feature_dir="data/google-gemma-3-12b-it-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="meta-llama/Llama-3.1-8B",
        feature_dir="data/meta-llama-Llama-3.1-8B-feature-act",
        sae_size=None,
    ),
]


# ============================================================
# Prompt templates
# ============================================================

def zero_shot_prompt2(text: str) -> str:
    return (
        "What is the inferred emotion in the following context?\n"
        f"Context: {text}.\n"
        "Answer:"
    )


def zero_shot_prompt3(text: str) -> str:
    return (
        "Consider this list of emotions: anger, disgust, fear, joy, sadness, surprise.\n"
        "What is the inferred emotion in the following context?\n"
        f"Context: {text}.\n"
        "Answer:"
    )


def zero_shot_prompt4(text: str) -> str:
    return f"Guess the emotion.\nContext: {text}.\nAnswer:"


PROMPT_FUNCTIONS: Dict[str, Callable[[str], str]] = {
    "prompt2": zero_shot_prompt2,
    "prompt3": zero_shot_prompt3,
    "prompt4": zero_shot_prompt4,
}


# ============================================================
# Helpers
# ============================================================

def validate_required_files(model_spec: ModelSpec) -> None:
    missing = [
        path
        for path in [
            model_spec.feature_csv_path,
            model_spec.best_checkpoint_path,
            model_spec.main_test_path,
            MAIN_TEST_PATH,
            MAIN_SOURCE_PATH,
        ]
        if not os.path.exists(path)
    ]
    if missing:
        formatted = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(
            f"Missing required files for {model_spec.model_name}:\n{formatted}"
        )


def load_external_test_dataframe(dataset_spec: Dict) -> pd.DataFrame:
    df = prepare_emotion_dataframe(
        csv_path=dataset_spec["path"],
        text_col=dataset_spec["text_col"],
        label_col=dataset_spec["label_col"],
        sep=dataset_spec["sep"],
        apply_prompt_template=False,
        keep_emotions=list(DEFAULT_EMOTIONS),
        drop_labels=dataset_spec["drop_labels"],
    )
    df = df.rename(columns={"input_text": "raw_text"})
    return df[["raw_text", "emotion"]].reset_index(drop=True)


def _normalize_saved_main_test(path: str) -> pd.DataFrame:
    """Read saved prompts for split comparison without discarding any rows."""
    df = pd.read_csv(path)
    required = {"input_text", "emotion"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {sorted(missing)} in {path}")

    df = df[["input_text", "emotion"]].copy()
    if df.empty or df.isna().any().any():
        raise ValueError(f"Empty split or missing values in {path}")
    df["input_text"] = df["input_text"].astype(str)
    df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
    if df["input_text"].str.strip().eq("").any() or not df["emotion"].isin(DEFAULT_EMOTIONS).all():
        raise ValueError(f"Blank input or unsupported emotion in {path}")
    return df.reset_index(drop=True)


def recover_main_raw_text(saved_df: pd.DataFrame, source_path: str) -> pd.DataFrame:
    """Recover original sentences by exact prompt/label match, preserving rows.

    Repeated identical source records are allowed. Missing or ambiguous matches
    fail explicitly; prompt-like words within original sentences are preserved.
    """
    source_df = pd.read_csv(source_path)
    keys = ["input_text", "emotion"]
    required = set(keys + ["text"])
    missing = required - set(source_df.columns)
    if missing:
        raise ValueError(f"Missing columns {sorted(missing)} in {source_path}")
    source_df = source_df[keys + ["text"]].copy()
    if source_df.isna().any().any():
        raise ValueError(f"Missing prompt, label, or raw text in {source_path}")
    source_df["emotion"] = source_df["emotion"].astype(str).str.lower().str.strip()
    source_df["input_text"] = source_df["input_text"].astype(str)
    source_df["text"] = source_df["text"].astype(str)
    if source_df["text"].str.strip().eq("").any():
        raise ValueError(f"Blank raw text in {source_path}")
    source_df = source_df.drop_duplicates(keys + ["text"])
    if source_df.duplicated(keys).any():
        raise ValueError(f"Ambiguous raw-text mapping for a prompt/label pair in {source_path}")

    lookup = source_df.set_index(keys)["text"].to_dict()
    saved_keys = list(saved_df[keys].itertuples(index=False, name=None))
    missing_count = sum(key not in lookup for key in saved_keys)
    if missing_count:
        raise ValueError(f"{missing_count} held-out rows have no exact raw-text match in {source_path}")
    return pd.DataFrame({
        "raw_text": [lookup[key] for key in saved_keys],
        "emotion": saved_df["emotion"].tolist(),
    })


def load_main_test_dataframe() -> pd.DataFrame:
    reference_df = _normalize_saved_main_test(MAIN_TEST_PATH)

    for model_spec in MODEL_SPECS:
        candidate_df = _normalize_saved_main_test(model_spec.main_test_path)
        if not reference_df.equals(candidate_df):
            raise ValueError(
                "The saved main-dataset holdout splits are not identical across "
                f"model runs. Reference: {MAIN_TEST_PATH}; "
                f"mismatch: {model_spec.main_test_path}"
            )

    return recover_main_raw_text(reference_df, MAIN_SOURCE_PATH)


def load_evaluation_datasets() -> Dict[str, pd.DataFrame]:
    datasets: Dict[str, pd.DataFrame] = {}

    for dataset_spec in EXTERNAL_TEST_DATASETS:
        name = dataset_spec["name"]
        if name in datasets:
            raise ValueError(f"Duplicate dataset name: {name}")
        datasets[name] = load_external_test_dataframe(dataset_spec)

    datasets[MAIN_TEST_DATASET_NAME] = load_main_test_dataframe()
    return datasets

def apply_prompt(raw_df: pd.DataFrame, prompt_fn: Callable[[str], str]) -> pd.DataFrame:
    prompted_df = raw_df.copy()
    prompted_df["input_text"] = prompted_df["raw_text"].map(prompt_fn)
    return prompted_df[["raw_text", "input_text", "emotion"]]


def calculate_metrics(
    y_true: List[str],
    y_pred: List[str],
    emotions: List[str],
) -> tuple[Dict[str, float], List[Dict[str, float]]]:
    report = classification_report(
        y_true,
        y_pred,
        labels=emotions,
        output_dict=True,
        zero_division=0,
    )

    overall = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=emotions,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=emotions,
                average="weighted",
                zero_division=0,
            )
        ),
    }

    per_emotion = []
    for emotion in emotions:
        emotion_metrics = report.get(emotion, {})
        per_emotion.append(
            {
                "emotion": emotion,
                "precision": float(emotion_metrics.get("precision", 0.0)),
                "recall": float(emotion_metrics.get("recall", 0.0)),
                "f1": float(emotion_metrics.get("f1-score", 0.0)),
                "support": int(emotion_metrics.get("support", 0)),
            }
        )

    return overall, per_emotion


def add_predictions(
    prediction_rows: List[Dict],
    prompted_df: pd.DataFrame,
    logits: np.ndarray,
    predictions: List[str],
    model_name: str,
    dataset_name: str,
    prompt_name: str,
    condition: str,
    emotions: List[str],
) -> None:
    for row_idx, row in prompted_df.reset_index(drop=True).iterrows():
        output_row = {
            "model": model_name,
            "dataset": dataset_name,
            "prompt": prompt_name,
            "condition": condition,
            "example_id": int(row_idx),
            "raw_text": row["raw_text"],
            "prompted_text": row["input_text"],
            "gold_emotion": row["emotion"],
            "pred_emotion": predictions[row_idx],
        }
        for emotion_idx, emotion in enumerate(emotions):
            output_row[f"logit_{emotion}"] = float(logits[row_idx, emotion_idx])
        prediction_rows.append(output_row)


def save_current_results(
    overall_rows: List[Dict],
    per_emotion_rows: List[Dict],
    prediction_rows: List[Dict],
) -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if overall_rows:
        safe_save_csv(
            pd.DataFrame(overall_rows),
            os.path.join(OUTPUT_DIR, "prompt_robustness_results_long.csv"),
        )
    if per_emotion_rows:
        safe_save_csv(
            pd.DataFrame(per_emotion_rows),
            os.path.join(OUTPUT_DIR, "prompt_robustness_per_emotion_long.csv"),
        )
    if SAVE_PREDICTIONS and prediction_rows:
        safe_save_csv(
            pd.DataFrame(prediction_rows),
            os.path.join(OUTPUT_DIR, "prompt_robustness_predictions.csv"),
        )


def create_comparison_tables(
    overall_df: pd.DataFrame,
    per_emotion_df: pd.DataFrame,
) -> None:
    metric_columns = ["accuracy", "macro_f1", "weighted_f1"]
    comparison = overall_df.pivot(
        index=["model", "dataset", "prompt", "n"],
        columns="condition",
        values=metric_columns,
    )
    comparison.columns = [f"{metric}_{condition}" for metric, condition in comparison.columns]
    comparison = comparison.reset_index()

    for metric in metric_columns:
        vanilla_col = f"{metric}_vanilla"
        steered_col = f"{metric}_steered"
        comparison[f"delta_{metric}"] = comparison[steered_col] - comparison[vanilla_col]

    safe_save_csv(
        comparison,
        os.path.join(OUTPUT_DIR, "prompt_robustness_comparison.csv"),
    )

    per_emotion_comparison = per_emotion_df.pivot(
        index=["model", "dataset", "prompt", "emotion", "support"],
        columns="condition",
        values=["precision", "recall", "f1"],
    )
    per_emotion_comparison.columns = [
        f"{metric}_{condition}"
        for metric, condition in per_emotion_comparison.columns
    ]
    per_emotion_comparison = per_emotion_comparison.reset_index()
    per_emotion_comparison["delta_f1"] = (
        per_emotion_comparison["f1_steered"]
        - per_emotion_comparison["f1_vanilla"]
    )

    safe_save_csv(
        per_emotion_comparison,
        os.path.join(OUTPUT_DIR, "prompt_robustness_per_emotion_comparison.csv"),
    )

    aggregate = (
        comparison.groupby(["model", "dataset"], as_index=False)
        .agg(
            num_prompts=("prompt", "nunique"),
            vanilla_macro_f1_mean=("macro_f1_vanilla", "mean"),
            steered_macro_f1_mean=("macro_f1_steered", "mean"),
            delta_macro_f1_mean=("delta_macro_f1", "mean"),
            vanilla_accuracy_mean=("accuracy_vanilla", "mean"),
            steered_accuracy_mean=("accuracy_steered", "mean"),
            delta_accuracy_mean=("delta_accuracy", "mean"),
        )
    )
    safe_save_csv(
        aggregate,
        os.path.join(OUTPUT_DIR, "prompt_robustness_aggregate.csv"),
    )


def cleanup_after_model(finetuner: Optional[SAEFeatureFinetuner]) -> None:
    if finetuner is not None:
        del finetuner
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ============================================================
# Main evaluation
# ============================================================

def evaluate_model(
    model_spec: ModelSpec,
    evaluation_datasets: Dict[str, pd.DataFrame],
    overall_rows: List[Dict],
    per_emotion_rows: List[Dict],
    prediction_rows: List[Dict],
) -> None:
    validate_required_files(model_spec)

    print("=" * 100)
    print(f"Model: {model_spec.model_name}")
    print(f"Feature CSV: {model_spec.feature_csv_path}")
    print(f"Best checkpoint: {model_spec.best_checkpoint_path}")
    print("=" * 100)

    finetuner = SAEFeatureFinetuner(
        model_name=model_spec.model_name,
        feature_csv_path=model_spec.feature_csv_path,
        emotions=list(DEFAULT_EMOTIONS),
        sae_size=model_spec.sae_size,
        cache_dir=CACHE_DIR,
        device=DEVICE,
        device_map=DEVICE_MAP,
        torch_dtype=TORCH_DTYPE,
        max_length=MAX_LENGTH,
    )

    try:
        for dataset_name, raw_test_df in evaluation_datasets.items():
            print(f"\nDataset: {dataset_name} (n={len(raw_test_df)})")

            for prompt_name, prompt_fn in PROMPT_FUNCTIONS.items():
                prompted_df = apply_prompt(raw_test_df, prompt_fn)

                # ------------------------------------------------
                # Vanilla model, no SAE intervention hooks
                # ------------------------------------------------
                y_true, vanilla_pred, vanilla_logits = finetuner.predict_clean(
                    prompted_df,
                    batch_size=EVAL_BATCH_SIZE,
                )
                vanilla_metrics, vanilla_per_emotion = calculate_metrics(
                    y_true,
                    vanilla_pred,
                    finetuner.emotions,
                )
                overall_rows.append(
                    {
                        "model": model_spec.model_name,
                        "dataset": dataset_name,
                        "prompt": prompt_name,
                        "condition": "vanilla",
                        "n": int(len(prompted_df)),
                        **vanilla_metrics,
                    }
                )
                for row in vanilla_per_emotion:
                    per_emotion_rows.append(
                        {
                            "model": model_spec.model_name,
                            "dataset": dataset_name,
                            "prompt": prompt_name,
                            "condition": "vanilla",
                            **row,
                        }
                    )
                if SAVE_PREDICTIONS:
                    add_predictions(
                        prediction_rows=prediction_rows,
                        prompted_df=prompted_df,
                        logits=vanilla_logits,
                        predictions=vanilla_pred,
                        model_name=model_spec.model_name,
                        dataset_name=dataset_name,
                        prompt_name=prompt_name,
                        condition="vanilla",
                        emotions=finetuner.emotions,
                    )

                # ------------------------------------------------
                # Best steered multipliers
                # ------------------------------------------------
                finetuner.load_state(model_spec.best_checkpoint_path)
                y_true_steered, steered_pred, steered_logits = finetuner.predict(
                    prompted_df,
                    batch_size=EVAL_BATCH_SIZE,
                )
                if y_true_steered != y_true:
                    raise RuntimeError("Vanilla and steered label ordering differs.")

                steered_metrics, steered_per_emotion = calculate_metrics(
                    y_true_steered,
                    steered_pred,
                    finetuner.emotions,
                )
                overall_rows.append(
                    {
                        "model": model_spec.model_name,
                        "dataset": dataset_name,
                        "prompt": prompt_name,
                        "condition": "steered",
                        "n": int(len(prompted_df)),
                        **steered_metrics,
                    }
                )
                for row in steered_per_emotion:
                    per_emotion_rows.append(
                        {
                            "model": model_spec.model_name,
                            "dataset": dataset_name,
                            "prompt": prompt_name,
                            "condition": "steered",
                            **row,
                        }
                    )
                if SAVE_PREDICTIONS:
                    add_predictions(
                        prediction_rows=prediction_rows,
                        prompted_df=prompted_df,
                        logits=steered_logits,
                        predictions=steered_pred,
                        model_name=model_spec.model_name,
                        dataset_name=dataset_name,
                        prompt_name=prompt_name,
                        condition="steered",
                        emotions=finetuner.emotions,
                    )

                delta_macro_f1 = (
                    steered_metrics["macro_f1"] - vanilla_metrics["macro_f1"]
                )
                print(
                    f"[{dataset_name} | {prompt_name}] "
                    f"vanilla macro-F1={vanilla_metrics['macro_f1']:.4f}, "
                    f"steered macro-F1={steered_metrics['macro_f1']:.4f}, "
                    f"delta={delta_macro_f1:+.4f}"
                )

                # Save after every prompt so completed results survive failures.
                save_current_results(
                    overall_rows,
                    per_emotion_rows,
                    prediction_rows,
                )

    finally:
        cleanup_after_model(finetuner)

In [ ]:
def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Validate paths before loading the shared main holdout split.
    for model_spec in MODEL_SPECS:
        validate_required_files(model_spec)

    evaluation_datasets = load_evaluation_datasets()

    for dataset_name, raw_test_df in evaluation_datasets.items():
        safe_save_csv(
            raw_test_df,
            os.path.join(OUTPUT_DIR, f"{dataset_name}_raw_test_df.csv"),
        )
        print(f"Dataset: {dataset_name}")
        print(f"Number of test examples: {len(raw_test_df)}")
        print("Label counts:")
        print(raw_test_df["emotion"].value_counts().sort_index())
        print()

    print(f"Prompts: {list(PROMPT_FUNCTIONS.keys())}")

    overall_rows: List[Dict] = []
    per_emotion_rows: List[Dict] = []
    prediction_rows: List[Dict] = []
    failures: List[str] = []

    for model_spec in MODEL_SPECS:
        try:
            evaluate_model(
                model_spec=model_spec,
                evaluation_datasets=evaluation_datasets,
                overall_rows=overall_rows,
                per_emotion_rows=per_emotion_rows,
                prediction_rows=prediction_rows,
            )
        except Exception as exc:
            message = f"FAILED {model_spec.model_name}: {exc!r}"
            print(message)
            traceback.print_exc()
            failures.append(message)
            cleanup_after_model(None)

    save_current_results(overall_rows, per_emotion_rows, prediction_rows)

    if overall_rows and per_emotion_rows:
        overall_df = pd.DataFrame(overall_rows)
        per_emotion_df = pd.DataFrame(per_emotion_rows)
        create_comparison_tables(overall_df, per_emotion_df)

    print("\nSaved outputs to:")
    print(OUTPUT_DIR)

    if failures:
        print("\nSome model evaluations failed:")
        for failure in failures:
            print(f"- {failure}")
    else:
        print("\nAll model evaluations finished successfully.")


if __name__ == "__main__":
    main()

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import pandas as pd


# ============================================================
# Paths and display order
# ============================================================

RESULT_DIR = Path("data/prompt_robustness_evaluation_raw_text")
PER_EMOTION_PATH = RESULT_DIR / "prompt_robustness_per_emotion_comparison.csv"
OVERALL_PATH = RESULT_DIR / "prompt_robustness_comparison.csv"
OUTPUT_DIR = RESULT_DIR / "paper_tables"

DATASET_NAME = "main_test"

MODEL_ORDER = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B",
]

MODEL_DISPLAY = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
    "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
}

PROMPT_ORDER = ["original", "prompt2", "prompt3", "prompt4"]

EMOTION_ORDER = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "macro",
]

EMOTION_DISPLAY = {
    "anger": "Anger",
    "disgust": "Disgust",
    "fear": "Fear",
    "joy": "Joy",
    "sadness": "Sadness",
    "surprise": "Surprise",
    "macro": "Macro-F1",
}


# ============================================================
# Original-prompt results reported in Table 1 of the paper
# ============================================================

# Each list follows:
# anger, disgust, fear, joy, sadness, surprise, macro_f1
ORIGINAL_PROMPT_RESULTS: Dict[str, Dict[str, List[float]]] = {
    "google/gemma-2-2b": {
        "vanilla": [0.54, 0.24, 0.71, 0.79, 0.73, 0.29, 0.55],
        "steered": [0.63, 0.68, 0.78, 0.81, 0.76, 0.45, 0.69],
    },
    "google/gemma-2-9b": {
        "vanilla": [0.62, 0.44, 0.77, 0.84, 0.74, 0.32, 0.62],
        "steered": [0.67, 0.70, 0.81, 0.86, 0.77, 0.58, 0.73],
    },
    "google/gemma-3-4b-pt": {
        "vanilla": [0.58, 0.36, 0.74, 0.71, 0.68, 0.39, 0.58],
        "steered": [0.64, 0.67, 0.77, 0.81, 0.76, 0.46, 0.69],
    },
    "google/gemma-3-4b-it": {
        "vanilla": [0.54, 0.42, 0.74, 0.83, 0.65, 0.27, 0.58],
        "steered": [0.66, 0.67, 0.79, 0.82, 0.77, 0.39, 0.68],
    },
    "google/gemma-3-12b-pt": {
        "vanilla": [0.65, 0.47, 0.78, 0.84, 0.79, 0.37, 0.65],
        "steered": [0.65, 0.72, 0.79, 0.85, 0.78, 0.46, 0.71],
    },
    "google/gemma-3-12b-it": {
        "vanilla": [0.53, 0.49, 0.77, 0.86, 0.68, 0.34, 0.61],
        "steered": [0.66, 0.68, 0.79, 0.83, 0.74, 0.48, 0.70],
    },
    "meta-llama/Llama-3.1-8B": {
        "vanilla": [0.65, 0.49, 0.75, 0.84, 0.77, 0.29, 0.63],
        "steered": [0.66, 0.66, 0.77, 0.83, 0.77, 0.44, 0.69],
    },
}


def validate_input_files() -> None:
    missing = [path for path in [PER_EMOTION_PATH, OVERALL_PATH] if not path.exists()]
    if missing:
        formatted = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(f"Missing prompt-robustness result files:\n{formatted}")


def build_original_prompt_rows() -> pd.DataFrame:
    rows = []

    for model in MODEL_ORDER:
        if model not in ORIGINAL_PROMPT_RESULTS:
            raise KeyError(f"Missing original-prompt results for {model}")

        vanilla_values = ORIGINAL_PROMPT_RESULTS[model]["vanilla"]
        steered_values = ORIGINAL_PROMPT_RESULTS[model]["steered"]

        for emotion, vanilla_f1, steered_f1 in zip(
            EMOTION_ORDER,
            vanilla_values,
            steered_values,
        ):
            rows.append(
                {
                    "model": model,
                    "prompt": "original",
                    "emotion": emotion,
                    "f1_vanilla": float(vanilla_f1),
                    "f1_steered": float(steered_f1),
                    "delta_f1": float(steered_f1 - vanilla_f1),
                }
            )

    return pd.DataFrame(rows)


def load_new_prompt_rows() -> pd.DataFrame:
    per_emotion_df = pd.read_csv(PER_EMOTION_PATH)
    overall_df = pd.read_csv(OVERALL_PATH)

    per_emotion_df = per_emotion_df[
        per_emotion_df["dataset"].eq(DATASET_NAME)
        & per_emotion_df["model"].isin(MODEL_ORDER)
        & per_emotion_df["prompt"].isin(["prompt2", "prompt3", "prompt4"])
    ].copy()

    overall_df = overall_df[
        overall_df["dataset"].eq(DATASET_NAME)
        & overall_df["model"].isin(MODEL_ORDER)
        & overall_df["prompt"].isin(["prompt2", "prompt3", "prompt4"])
    ].copy()

    emotion_rows = per_emotion_df[
        [
            "model",
            "prompt",
            "emotion",
            "f1_vanilla",
            "f1_steered",
            "delta_f1",
        ]
    ].copy()

    macro_rows = overall_df[
        [
            "model",
            "prompt",
            "macro_f1_vanilla",
            "macro_f1_steered",
            "delta_macro_f1",
        ]
    ].copy()

    macro_rows = macro_rows.rename(
        columns={
            "macro_f1_vanilla": "f1_vanilla",
            "macro_f1_steered": "f1_steered",
            "delta_macro_f1": "delta_f1",
        }
    )
    macro_rows["emotion"] = "macro"

    return pd.concat([emotion_rows, macro_rows], ignore_index=True)


def validate_complete_grid(df: pd.DataFrame) -> None:
    expected = {
        (model, prompt, emotion)
        for model in MODEL_ORDER
        for prompt in PROMPT_ORDER
        for emotion in EMOTION_ORDER
    }
    observed = set(
        df[["model", "prompt", "emotion"]]
        .itertuples(index=False, name=None)
    )

    missing = expected - observed
    duplicates = df.duplicated(["model", "prompt", "emotion"], keep=False)

    if missing:
        preview = sorted(missing)[:20]
        raise ValueError(
            "Missing model-prompt-emotion rows. "
            f"Examples: {preview}"
        )

    if duplicates.any():
        duplicate_rows = df.loc[
            duplicates,
            ["model", "prompt", "emotion"],
        ]
        raise ValueError(
            "Duplicate result rows found:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )


def average_across_prompts(all_rows: pd.DataFrame) -> pd.DataFrame:
    """Average Original, Prompt 2, Prompt 3, and Prompt 4 for each model."""
    summary = (
        all_rows.groupby(["model", "emotion"], as_index=False)
        .agg(
            vanilla_f1_mean=("f1_vanilla", "mean"),
            vanilla_f1_sd=("f1_vanilla", "std"),
            steered_f1_mean=("f1_steered", "mean"),
            steered_f1_sd=("f1_steered", "std"),
            delta_f1_mean=("delta_f1", "mean"),
            delta_f1_sd=("delta_f1", "std"),
            num_prompts=("prompt", "nunique"),
            num_prompts_improved=("delta_f1", lambda x: int((x > 0).sum())),
            num_prompts_not_harmed=("delta_f1", lambda x: int((x >= 0).sum())),
            min_delta_across_prompts=("delta_f1", "min"),
            max_delta_across_prompts=("delta_f1", "max"),
        )
    )

    summary["model"] = pd.Categorical(
        summary["model"],
        categories=MODEL_ORDER,
        ordered=True,
    )
    summary["emotion"] = pd.Categorical(
        summary["emotion"],
        categories=EMOTION_ORDER,
        ordered=True,
    )

    return summary.sort_values(["model", "emotion"]).reset_index(drop=True)


def make_numeric_table(summary: pd.DataFrame) -> pd.DataFrame:
    """Two rows per model: vanilla average and steered average."""
    rows = []

    for model in MODEL_ORDER:
        model_df = summary[summary["model"].eq(model)].set_index("emotion")

        for setting, value_col in [
            ("Vanilla", "vanilla_f1_mean"),
            ("Steered", "steered_f1_mean"),
        ]:
            row = {
                "Model": MODEL_DISPLAY[model],
                "Setting": setting,
            }
            for emotion in EMOTION_ORDER:
                row[EMOTION_DISPLAY[emotion]] = float(
                    model_df.loc[emotion, value_col]
                )
            rows.append(row)

    return pd.DataFrame(rows)


def make_transition_table(summary: pd.DataFrame) -> pd.DataFrame:
    """One row per model, with each cell showing vanilla to steered and delta."""
    rows = []

    for model in MODEL_ORDER:
        model_df = summary[summary["model"].eq(model)].set_index("emotion")
        row = {"Model": MODEL_DISPLAY[model]}

        for emotion in EMOTION_ORDER:
            vanilla = float(model_df.loc[emotion, "vanilla_f1_mean"])
            steered = float(model_df.loc[emotion, "steered_f1_mean"])
            delta = float(model_df.loc[emotion, "delta_f1_mean"])
            improved = int(model_df.loc[emotion, "num_prompts_improved"])
            total = int(model_df.loc[emotion, "num_prompts"])

            row[EMOTION_DISPLAY[emotion]] = (
                f"{vanilla:.3f} to {steered:.3f} "
                f"({delta:+.3f}; {improved}/{total})"
            )

        rows.append(row)

    return pd.DataFrame(rows)


def make_delta_consistency_table(summary: pd.DataFrame) -> pd.DataFrame:
    """One row per model, with mean delta and number of prompts improved."""
    rows = []

    for model in MODEL_ORDER:
        model_df = summary[summary["model"].eq(model)].set_index("emotion")
        row = {"Model": MODEL_DISPLAY[model]}

        for emotion in EMOTION_ORDER:
            delta = float(model_df.loc[emotion, "delta_f1_mean"])
            improved = int(model_df.loc[emotion, "num_prompts_improved"])
            total = int(model_df.loc[emotion, "num_prompts"])
            minimum = float(model_df.loc[emotion, "min_delta_across_prompts"])

            row[EMOTION_DISPLAY[emotion]] = (
                f"{delta:+.3f} ({improved}/{total}; min {minimum:+.3f})"
            )

        rows.append(row)

    return pd.DataFrame(rows)


def dataframe_to_latex(
    df: pd.DataFrame,
    path: Path,
    numeric: bool,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    if numeric:
        latex = df.to_latex(
            index=False,
            float_format=lambda x: f"{x:.3f}",
            escape=False,
            multicolumn=False,
            column_format="ll" + "c" * (len(df.columns) - 2),
        )
    else:
        latex = df.to_latex(
            index=False,
            escape=False,
            multicolumn=False,
            column_format="l" + "c" * (len(df.columns) - 1),
        )

    path.write_text(latex, encoding="utf-8")


def main() -> None:
    validate_input_files()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    original_rows = build_original_prompt_rows()
    new_prompt_rows = load_new_prompt_rows()
    all_rows = pd.concat(
        [original_rows, new_prompt_rows],
        ignore_index=True,
    )

    validate_complete_grid(all_rows)

    all_rows["prompt"] = pd.Categorical(
        all_rows["prompt"],
        categories=PROMPT_ORDER,
        ordered=True,
    )
    all_rows["emotion"] = pd.Categorical(
        all_rows["emotion"],
        categories=EMOTION_ORDER,
        ordered=True,
    )
    all_rows = all_rows.sort_values(
        ["model", "prompt", "emotion"]
    ).reset_index(drop=True)

    summary = average_across_prompts(all_rows)
    numeric_table = make_numeric_table(summary)
    transition_table = make_transition_table(summary)
    consistency_table = make_delta_consistency_table(summary)

    all_rows.to_csv(
        OUTPUT_DIR / "main_prompt_all_prompt_model_level_f1.csv",
        index=False,
    )
    summary.to_csv(
        OUTPUT_DIR / "main_prompt_average_all_prompts_per_model_long.csv",
        index=False,
    )
    numeric_table.to_csv(
        OUTPUT_DIR / "main_prompt_average_all_prompts_per_model_table.csv",
        index=False,
    )
    transition_table.to_csv(
        OUTPUT_DIR / "main_prompt_average_all_prompts_transition_table.csv",
        index=False,
    )
    consistency_table.to_csv(
        OUTPUT_DIR / "main_prompt_average_all_prompts_consistency_table.csv",
        index=False,
    )

    dataframe_to_latex(
        numeric_table,
        OUTPUT_DIR / "main_prompt_average_all_prompts_per_model_table.tex",
        numeric=True,
    )
    dataframe_to_latex(
        transition_table,
        OUTPUT_DIR / "main_prompt_average_all_prompts_transition_table.tex",
        numeric=False,
    )
    dataframe_to_latex(
        consistency_table,
        OUTPUT_DIR / "main_prompt_average_all_prompts_consistency_table.tex",
        numeric=False,
    )

    print("\nAverage across all four prompts, shown per model")
    print("=" * 120)
    print(
        numeric_table.to_string(
            index=False,
            float_format=lambda x: f"{x:.3f}",
        )
    )

    print("\nCompact transition table")
    print("Each cell is vanilla to steered, mean delta, and prompts improved.")
    print("=" * 120)
    print(transition_table.to_string(index=False))

    print(f"\nSaved tables to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()